In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from src import evaluare as ev

FISIER = "../data/processed/cipro_norvegia.tsv"   # pentru ampicilina: ../data/processed/ampicillin_norvegia.tsv
ETICHETA = "cipro"                                  # pentru ampicilina: "ampicillin"
N_REP = 5        # repetari pentru impartirile aleatorii si pe clustere (creste la 10-20 mai tarziu)
N_CTRL = 100     # repetari pentru controlul cu distanta potrivita

X = sparse.load_npz("../data/processed/kmeri_X.npz")
ids = pd.read_csv("../data/processed/kmeri_ids.txt", header=None, dtype=str)[0].tolist()
id2row = {g: i for i, g in enumerate(ids)}
df = ev.incarca_date(FISIER)
df = df[df["genome_id"].isin(id2row)].reset_index(drop=True)
print(df.shape, X.shape, round(X.nnz / X.shape[0]), "hash-uri medii per genom")

(3148, 14) (3213, 74527) 5016 hash-uri medii per genom


In [2]:
#verificam descarcarea
log = pd.read_csv("../data/processed/schite_log.csv")
print(log["stare"].value_counts())
log[log["stare"] == "ok"][["contiguri", "lungime", "hash_uri"]].describe().round(0)

stare
ok        3210
exista       3
Name: count, dtype: int64


,contiguri,lungime,hash_uri
count,3210.0,3210.0,3210.0
mean,114.0,5075164.0,5016.0
std,44.0,160236.0,162.0
min,21.0,4420089.0,4395.0
25%,83.0,4991017.0,4924.0
50%,107.0,5089932.0,5016.0
75%,135.0,5175674.0,5111.0
max,353.0,5746287.0,5650.0


In [3]:
#modelele si evaluarea temporala
MODELE_COMP = {
    "linie ST": ev.fabrica_linie("ST"),
    "linie HC50": ev.fabrica_linie("HC50"),
    "linie HC100": ev.fabrica_linie("HC100"),
    "k-meri (reg. logistica)": ev.fabrica_kmeri(X, id2row, C=0.1, min_df=5),
}
train_t, test_t = ev.split_temporal(df, 2012)
rez = {nume: ev.metrici(test_t["y"].values, fn(train_t, test_t)[0]) for nume, fn in MODELE_COMP.items()}
pd.DataFrame(rez).T.round(3)

,AUROC,AUPRC_castig
linie ST,0.877,0.445
linie HC50,0.857,0.509
linie HC100,0.897,0.573
k-meri (reg. logistica),0.917,0.597


In [4]:
#aleatoriu fata de temporal
n_tr, n_te = len(train_t), len(test_t)
aleatoriu = {n: [] for n in MODELE_COMP}
for seed in range(N_REP):
    tr, te = ev.split_aleatoriu(df, n_tr, n_te, seed)
    for nume, fn in MODELE_COMP.items():
        aleatoriu[nume].append(ev.metrici(te["y"].values, fn(tr, te)[0])["AUROC"])
pd.DataFrame({
    "AUROC aleatoriu": {n: f"{np.mean(v):.3f} +- {np.std(v, ddof=1):.3f}" for n, v in aleatoriu.items()},
    "AUROC temporal": {n: round(rez[n]["AUROC"], 3) for n in rez}})

,AUROC aleatoriu,AUROC temporal
linie ST,0.904 +- 0.010,0.877
linie HC50,0.899 +- 0.010,0.857
linie HC100,0.922 +- 0.012,0.897
k-meri (reg. logistica),0.945 +- 0.007,0.917


In [5]:
#impartiri pe clustere (curba performanta-distanta pentru fiecare model)
frac = len(test_t) / len(df)
niveluri_split = ["aleatoriu", 10, 20, 50, 100]
rez_cl = {nume: {} for nume in MODELE_COMP}
for nivel in niveluri_split:
    sc = {n: [] for n in MODELE_COMP}
    for seed in range(N_REP):
        if nivel == "aleatoriu":
            tr, te = ev.split_aleatoriu(df, len(df) - len(test_t), len(test_t), seed)
        else:
            tr, te = ev.split_pe_clustere(df, nivel, frac, seed)
        if te["y"].nunique() < 2 or tr["y"].nunique() < 2:
            continue
        for nume, fn in MODELE_COMP.items():
            sc[nume].append(ev.metrici(te["y"].values, fn(tr, te)[0])["AUROC"])
    for nume in MODELE_COMP:
        rez_cl[nume][nivel] = np.nanmean(sc[nume])
tab_cl = pd.DataFrame(rez_cl).T.round(3)
tab_cl.columns = ["aleatoriu" if c == "aleatoriu" else f"HC{c}" for c in tab_cl.columns]
tab_cl

,aleatoriu,HC10,HC20,HC50,HC100
linie ST,0.904,0.886,0.881,0.752,0.687
linie HC50,0.899,0.878,0.863,0.500,0.500
linie HC100,0.922,0.911,0.899,0.713,0.500
k-meri (reg. logistica),0.945,0.937,0.923,0.847,0.800


In [6]:
#controlul cu distanta potrivita, k-meri fata de linie
fn_k = MODELE_COMP["k-meri (reg. logistica)"]
rez_ck = pd.DataFrame([r for r in (ev.control_distanta(df, seed=s, model_fn=fn_k) for s in range(N_CTRL)) if r])
rez_l50 = pd.DataFrame([r for r in (ev.control_distanta(df, seed=s, nume_model="HC50") for s in range(N_CTRL)) if r])
coloane = ["AUROC_temporal", "AUROC_control", "delta_AUROC"]

def rezumat_ic(r):
    return pd.DataFrame({"medie": r[coloane].mean(), "IC95_jos": r[coloane].quantile(0.025),
                         "IC95_sus": r[coloane].quantile(0.975)}).round(3)

print("k-meri");     display(rezumat_ic(rez_ck))
print("linie HC50"); display(rezumat_ic(rez_l50))

k-meri


,medie,IC95_jos,IC95_sus
AUROC_temporal,0.908,0.856,0.935
AUROC_control,0.965,0.906,0.997
delta_AUROC,-0.057,-0.110,-0.009


linie HC50


,medie,IC95_jos,IC95_sus
AUROC_temporal,0.840,0.801,0.872
AUROC_control,0.863,0.780,0.936
delta_AUROC,-0.023,-0.104,0.079
